# ⚡ High-Speed Cloud Torrent Downloader (Google Colab)
Download any torrent using Google Cloud's 1 Gbps datacenter network without consuming your home data or storage.

---
### 📌 How to Use:
1. **Option 1 (Full Web UI)**: Run Cell 1 below to launch a modern Web Torrent dashboard (qBittorrent) with a public Cloudflare Tunnel link.
2. **Option 2 (Direct 1-Click Link)**: Run Cell 2 to paste a magnet link and instantly get a direct HTTP download link for IDM/browser.


## 🌐 Option 1: Launch Full Web Torrent Dashboard (qBittorrent + Cloudflare)
Run this cell to start qBittorrent and get a secure public Web UI URL.

In [ ]:
#@title 🚀 Start Web UI Dashboard
import subprocess, time, os

print("📦 Installing qBittorrent and Cloudflare Tunnel...")
!apt-get -qq update > /dev/null
!apt-get -qq install -y qbittorrent-nox > /dev/null
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Setup download directory
os.makedirs("/content/downloads", exist_ok=True)

# Setup qBittorrent configuration
os.makedirs("/root/.config/qBittorrent", exist_ok=True)
conf = """[BitTorrent]
Session\\DefaultSavePath=/content/downloads
Session\\Port=6881
Session\\TempPath=/content/downloads/temp

[LegalNotice]
Accepted=true

[Preferences]
WebUI\\Address=*
WebUI\\Port=8080
WebUI\\AuthSubnetWhitelistEnabled=true
WebUI\\AuthSubnetWhitelist=0.0.0.0/0
WebUI\\LocalHostAuth=false
"""
with open("/root/.config/qBittorrent/qBittorrent.conf", "w") as f:
    f.write(conf)

# Start qBittorrent daemon
subprocess.Popen(["qbittorrent-nox", "-d"])
time.sleep(2)

print("\n" + "="*60)
print("⚡ Generating your secure Web UI link...")
print("="*60 + "\n")

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8080"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if "trycloudflare.com" in line:
        for token in line.split():
            if "trycloudflare.com" in token:
                print(f"👉 CLICK HERE TO OPEN YOUR TORRENT DASHBOARD:\n🔗 {token}\n")
                break
        break


## ⚡ Option 2: 1-Click Form Downloader (Fast Aria2 + GoFile / Google Drive)
Paste any Magnet Link or Torrent URL below and click the **Play** button:

In [ ]:
#@title 📥 Download Torrent & Generate Direct Link { display-mode: "form" }
Magnet_Link = "" #@param {type:"string"}
Upload_To = "GoFile CDN (Fast Direct Link for IDM)" #@param ["GoFile CDN (Fast Direct Link for IDM)", "Save Directly to Google Drive"]

import os, glob, subprocess

if not Magnet_Link.strip():
    print("⚠️ Please paste a valid Magnet Link or Torrent URL in the field above!")
else:
    # Install aria2
    print("📦 Setting up high-speed aria2...")
    os.system("apt-get -qq update > /dev/null && apt-get -qq install -y aria2 curl > /dev/null")
    os.makedirs("/content/downloads", exist_ok=True)

    # Fetch best active trackers list
    try:
        trackers_raw = subprocess.check_output(["curl", "-s", "https://raw.githubusercontent.com/ngosang/trackerslist/master/trackers_best.txt"], text=True)
        trackers = ",".join([t.strip() for t in trackers_raw.splitlines() if t.strip()])
    except:
        trackers = ""

    print("⚡ Downloading torrent on Google Cloud 1 Gbps network...")
    cmd = [
        "aria2c",
        "--dir=/content/downloads",
        "--max-connection-per-server=16",
        "--split=16",
        "--seed-time=0",
        "--summary-interval=5",
        "--peer-id-prefix=-TR3000-",
        "--user-agent=Transmission/3.00"
    ]
    if trackers:
        cmd.append(f"--bt-tracker={trackers}")
    cmd.append(Magnet_Link.strip())

    subprocess.run(cmd)
    print("\n✅ Download complete on cloud VM!")

    # Find largest file or downloaded files
    files = [f for f in glob.glob("/content/downloads/**/*", recursive=True) if os.path.isfile(f) and not f.endswith(".aria2")]

    if files:
        largest_file = max(files, key=os.path.getsize)
        size_mb = os.path.getsize(largest_file) / (1024 * 1024)
        print(f"\n📁 Ready: {os.path.basename(largest_file)} ({size_mb:.2f} MB)")

        if Upload_To.startswith("GoFile"):
            print("☁️ Uploading to GoFile CDN for direct IDM link...")
            res = subprocess.check_output(["curl", "-s", "-F", f"file=@{largest_file}", "https://store1.gofile.io/contents/uploadfile"], text=True)
            print("\n" + "="*60)
            print(f"🎉 GOFILE DIRECT DOWNLOAD LINK:\n{res}")
            print("="*60)
        else:
            from google.colab import drive
            drive.mount("/content/drive")
            dest_dir = "/content/drive/My Drive/Torrents"
            os.makedirs(dest_dir, exist_ok=True)
            os.system(f'cp -rv "{largest_file}" "{dest_dir}"')
            print(f"🎉 File saved to Google Drive: Torrents/{os.path.basename(largest_file)}")
    else:
        print("⚠️ No downloaded files found.")


## ☁️ Option 3: Upload files downloaded from Web UI to GoFile / Google Drive
If you used Option 1 (Web UI) to download, run this cell to get a direct high-speed HTTP link for IDM.

In [ ]:
#@title 📤 Upload Downloaded Files to GoFile
import glob, os, subprocess

files = [f for f in glob.glob("/content/downloads/**/*", recursive=True) if os.path.isfile(f) and not f.endswith(".aria2")]
if not files:
    print("⚠️ No downloaded files found in /content/downloads.")
else:
    print(f"Found {len(files)} file(s):")
    for f in files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f" - {os.path.basename(f)} ({size_mb:.2f} MB)")
        print(f"   Uploading to GoFile...")
        os.system(f'curl -F "file=@{f}" https://store1.gofile.io/contents/uploadfile')
        print("\n")
